# az-sentiment: Colab-da işə salma
Runtime → Change runtime type → **T4 GPU**. Hücrələri ardıcıl işlət.

## 1. Kodu gətir
GitHub repo varsa `REPO_URL`-u doldur, yoxdursa lokal `az-sentiment.zip`-i yüklə (data/, models/, .venv/ olmadan).

In [ ]:
REPO_URL = ""  # məs: "https://github.com/<user>/az-sentiment.git" (private-dirsə token lazımdır)
import os
if REPO_URL:
    !git clone -q $REPO_URL /content/az-sentiment
else:
    from google.colab import files
    up = files.upload()  # az-sentiment.zip
    !unzip -q -o az-sentiment.zip -d /content/
%cd /content/az-sentiment
!ls

## 2. Asılılıqlar
Colab-da torch artıq var, qalanı quraşdırılır.

In [ ]:
!pip install -q -e . tabulate
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Data + baseline + TF-IDF

In [ ]:
!python scripts/01_prepare_data.py
!python scripts/02_baselines.py
!python scripts/03_tfidf.py

## 4. XLM-R fine-tune
T4-də bf16 yoxdur, skript avtomatik fp16-ya keçir. Hər run ~35–45 dəq.

In [ ]:
!python scripts/04_transformer.py --weighted
!python scripts/04_transformer.py

## 5. LLM (Ollama + deepseek-r1:1.5b, Colab daxilində)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull deepseek-r1:1.5b

In [ ]:
!python scripts/05_llm.py --shots 0 --n 500
!python scripts/05_llm.py --shots 3 --n 500

## 6. Müqayisə və səhv təhlili

In [ ]:
!python scripts/06_compare.py
!python scripts/07_error_analysis.py --model xlm-roberta-base_weighted --other tfidf_logreg

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison.md", encoding="utf-8").read()))
for f in ["eda", "comparison", "learning_curve_tfidf", "history_xlm-roberta-base_weighted", "cm_xlm-roberta-base_weighted"]:
    display(Image(f"results/figures/{f}.png"))

## 7. Nəticələri endir
`results/` (metrikalar, qrafiklər, səhv təhlili) zip olaraq. Model çəkiləri böyükdür, lazım olsa Drive-a köçür.

In [ ]:
!zip -q -r results.zip results
from google.colab import files
files.download("results.zip")